<a href="https://colab.research.google.com/github/selooo0/gulf-ticket-assistant/blob/main/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/SDAIAAcademy/applied-nlp.git"
if not Path("/content/applied-nlp").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/applied-nlp"], check=True)
os.chdir("/content/applied-nlp/labs")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
if not Path("data/tickets_train.csv").exists():
    subprocess.run([sys.executable, "prepare_data.py"], check=True)
sys.path.insert(0, os.getcwd())
import nlp_utils as U
U.set_seed(42)

from google.colab import drive
drive.mount("/content/drive")
PROJ = "/content/drive/MyDrive/ticket_assistant"
print("device:", U.device())

Mounted at /content/drive
device: cuda


In [2]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

train = U.load_tickets("train")                                  # إنجليزي (الكورس)
test  = U.load_tickets("test")                                   # اختبار إنجليزي
g_tr  = pd.read_csv(f"{PROJ}/gulf_train_split.csv")              # خليجي تدريب (147)
g_val = pd.read_csv(f"{PROJ}/gulf_val.csv")                      # خليجي validation (40)
gulf  = pd.read_csv(f"{PROJ}/NLP_Intent_Classification.csv")     # اختبار خليجي يدوي (40)
L = sorted(gulf.intent.unique())

# الخليجي قليل جداً مقارنة بالإنجليزي، فنكرره 10 مرات عشان النموذج ينتبه له
ft = pd.concat([train[["text", "intent"]]] + [g_tr[["text", "intent"]]] * 10).sample(frac=1, random_state=42)
labels = sorted(ft.intent.unique())
lab2id = {l: i for i, l in enumerate(labels)}
print(f"EN: {len(train)} | Gulf: {len(g_tr)} x10 | المجموع: {len(ft)} | عدد الأنواع: {len(labels)}")

EN: 5400 | Gulf: 147 x10 | المجموع: 6870 | عدد الأنواع: 27


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL = "distilbert-base-multilingual-cased"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=lab2id)

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, ys):
        self.enc = tok(list(texts), truncation=True, max_length=64, padding="max_length")
        self.ys = [lab2id[y] for y in ys]
    def __len__(self): return len(self.ys)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.ys[i])
        return item

args = TrainingArguments(output_dir="/content/ft_out", num_train_epochs=4, learning_rate=5e-5,
                         per_device_train_batch_size=32, fp16=torch.cuda.is_available(),
                         save_strategy="no", logging_steps=100, report_to="none", seed=42)
trainer = Trainer(model=model, args=args, train_dataset=DS(ft.text, ft.intent))
trainer.train()

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,2.181287
200,0.447364
300,0.071522
400,0.031783
500,0.017533
600,0.009315
700,0.005960
800,0.005395


TrainOutput(global_step=860, training_loss=0.3224882429768873, metrics={'train_runtime': 61.9916, 'train_samples_per_second': 443.286, 'train_steps_per_second': 13.873, 'total_flos': 455228382735360.0, 'train_loss': 0.3224882429768873, 'epoch': 4.0})

In [4]:
@torch.no_grad()
def predict(texts, bs=64):
    model.eval(); preds, confs = [], []
    for i in range(0, len(texts), bs):
        b = tok(list(texts[i:i+bs]), truncation=True, max_length=64, padding=True, return_tensors="pt").to(model.device)
        p = torch.softmax(model(**b).logits, -1)
        confs += p.max(-1).values.tolist(); preds += [labels[j] for j in p.argmax(-1).tolist()]
    return np.array(preds), np.array(confs)

p_test, c_test = predict(gulf.text.tolist())
p_val, _ = predict(g_val.text.tolist())
p_en, _ = predict(test.text.tolist())

ft_res = pd.DataFrame([
    {"model": "e5 + LR + Gulf (main pipeline)", "Gulf test": 0.791, "Gulf val": 0.734, "EN test": 0.995},
    {"model": "DistilmBERT fine-tuned + Gulf",
     "Gulf test": f1_score(gulf.intent, p_test, labels=L, average="macro"),
     "Gulf val":  f1_score(g_val.intent, p_val, labels=L, average="macro"),
     "EN test":   f1_score(test.intent, p_en, average="macro")}]).set_index("model").round(3)
ft_res

,Gulf test,Gulf val,EN test
model,,,
e5 + LR + Gulf (main pipeline),0.791,0.734,0.995
DistilmBERT fine-tuned + Gulf,0.729,0.712,0.996


In [5]:
ft_res.to_csv(f"{PROJ}/fine_tuning_results.csv")
pd.DataFrame({"text": gulf.text, "true": gulf.intent, "pred": p_test, "conf": c_test.round(3)}) \
  .to_csv(f"{PROJ}/fine_tuning_test_preds.csv", index=False)
print("تم الحفظ ✅")

تم الحفظ ✅
